# Project 3: Tumor Shape and Genetics in Gliomas
## Step 2 (Kaggle version): Train a U-Net on all 5 folds

This notebook builds the slice table, trains one U-Net per fold, and saves a predicted tumor outline for every patient.

**Setup on Kaggle**
1. Click **Add Input** in the right panel, search for **lgg-mri-segmentation** (by Mateusz Buda), and add it.
2. Open **Settings** in the right panel and set **Accelerator** to a GPU (GPU T4 x2 or GPU P100).
3. To run it in the background, click **Save Version**, choose **Save & Run All (Commit)**, and click Save. You can close the tab while it runs.
4. When it finishes, open the saved version and go to the **Output** tab to download `project3_outputs.zip`.

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F

plt.rcParams["font.size"] = 20

# use the Mac's graphics chip if it has one, otherwise the regular processor
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print("Training on:", device)

### Settings
`folds_to_run` is set to all 5 folds, so every patient gets a prediction from a model that never saw them during training.

In [ ]:
epochs = 30
batch_size = 16
learning_rate = 0.001
seed = 42

num_folds = 5
folds_to_run = [0, 1, 2, 3, 4]

torch.manual_seed(seed)
np.random.seed(seed)

### Find the dataset and build the slice table
This is the same as step 1. It finds the copy of the data that has the spreadsheet, pairs every image with its mask, and adds each patient's genetic groups.

In [ ]:
import glob

input_root = "/kaggle/input"
csv_files = sorted(glob.glob(os.path.join(input_root, "**", "data.csv"), recursive=True))
data_dir = os.path.dirname(csv_files[0])
print("Using data in:", data_dir)

mask_files = sorted(glob.glob(os.path.join(data_dir, "**", "*_mask.tif"), recursive=True))

# keep only one copy of each file, in case the folder holds duplicates
seen = set()
unique_masks = []
for f in mask_files:
    name = os.path.basename(f)
    if name not in seen:
        seen.add(name)
        unique_masks.append(f)
mask_files = unique_masks

rows = []
for mask_file in mask_files:
    image_file = mask_file.replace("_mask.tif", ".tif")
    folder = os.path.basename(os.path.dirname(image_file))
    patient = "_".join(folder.split("_")[:3])
    slice_num = int(os.path.basename(image_file).replace(".tif", "").split("_")[-1])
    rows.append({"Patient": patient, "Folder": folder, "Slice": slice_num,
                 "image_file": image_file, "mask_file": mask_file})

slices = pd.DataFrame(rows)
genetics = pd.read_csv(csv_files[0])
genetics["Patient"] = genetics[genetics.columns[0]].astype(str)
slices = slices.merge(genetics, on="Patient", how="left")
slices = slices.sort_values(["Patient", "Slice"]).reset_index(drop=True)
slices.to_csv("slices_with_genetics2.csv", index=False)

print("Patients:", slices["Patient"].nunique())
print("Slices:", len(slices))

### Load every image and mask into memory
Each image has 3 channels (pre-contrast, FLAIR, post-contrast). Pixel values are scaled to between 0 and 1. This takes a few minutes and uses under 1 GB of memory.

In [ ]:
num_slices = len(slices)
images = np.zeros((num_slices, 3, 256, 256), dtype=np.uint8)
masks = np.zeros((num_slices, 1, 256, 256), dtype=np.uint8)

start = time.time()
for i, row in slices.iterrows():
    image = np.array(Image.open(row["image_file"]))
    mask = np.array(Image.open(row["mask_file"]))
    images[i] = image.transpose(2, 0, 1)
    masks[i, 0] = (mask > 0).astype(np.uint8)
    if i % 500 == 0:
        print("Loaded", i, "of", num_slices)

print("Done in", round(time.time() - start), "seconds")

### Split patients into 5 groups
The split is by patient, not by slice. If slices from the same patient ended up in both training and testing, the model would be tested on a brain it had already seen, and the scores would look better than they really are.

In [ ]:
patients = np.array(sorted(slices["Patient"].unique()))
rng = np.random.default_rng(seed)
rng.shuffle(patients)
patient_groups = np.array_split(patients, num_folds)

for k, group in enumerate(patient_groups):
    print("Fold", k, ":", len(group), "patients")

### The U-Net
Same idea as your dissertation model. The left side shrinks the image to pick up patterns, and the right side grows it back to full size to draw the outline. The skip connections carry fine detail straight across so the edges stay sharp.

In [ ]:
class double_conv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class unet(nn.Module):
    def __init__(self, in_channels=3, base=16):
        super().__init__()
        self.down1 = double_conv(in_channels, base)
        self.down2 = double_conv(base, base * 2)
        self.down3 = double_conv(base * 2, base * 4)
        self.down4 = double_conv(base * 4, base * 8)
        self.bottom = double_conv(base * 8, base * 16)

        self.up4 = nn.ConvTranspose2d(base * 16, base * 8, 2, stride=2)
        self.conv4 = double_conv(base * 16, base * 8)
        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.conv3 = double_conv(base * 8, base * 4)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.conv2 = double_conv(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.conv1 = double_conv(base * 2, base)

        self.out = nn.Conv2d(base, 1, 1)

    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(F.max_pool2d(d1, 2))
        d3 = self.down3(F.max_pool2d(d2, 2))
        d4 = self.down4(F.max_pool2d(d3, 2))
        b = self.bottom(F.max_pool2d(d4, 2))

        u4 = self.conv4(torch.cat([self.up4(b), d4], dim=1))
        u3 = self.conv3(torch.cat([self.up3(u4), d3], dim=1))
        u2 = self.conv2(torch.cat([self.up2(u3), d2], dim=1))
        u1 = self.conv1(torch.cat([self.up1(u2), d1], dim=1))
        return self.out(u1)


model = unet()
print("Parameters:", sum(p.numel() for p in model.parameters()))

### Loss, prediction, and scoring
The loss mixes two things. Binary cross-entropy checks every pixel, and Dice loss checks how well the overall outline overlaps.

The Dice score is calculated per patient, by adding up the overlap across all of that patient's slices. That's how the Buda paper reports it, so your numbers will be directly comparable.

In [ ]:
def dice_loss(logits, targets):
    probs = torch.sigmoid(logits)
    overlap = (probs * targets).sum()
    total = probs.sum() + targets.sum()
    return 1 - (2 * overlap + 1) / (total + 1)


def predict(model, image_array):
    # returns a 0/1 tumor mask for every slice in image_array
    model.eval()
    predictions = np.zeros((len(image_array), 1, 256, 256), dtype=np.uint8)
    with torch.no_grad():
        for i in range(0, len(image_array), batch_size):
            x = torch.from_numpy(image_array[i:i + batch_size]).float().div(255).to(device)
            probs = torch.sigmoid(model(x)).cpu().numpy()
            predictions[i:i + batch_size] = (probs > 0.5).astype(np.uint8)
    return predictions


def dice_per_patient(predictions, true_masks, patient_names):
    scores = {}
    for name in np.unique(patient_names):
        rows = patient_names == name
        p = predictions[rows].astype(bool)
        t = true_masks[rows].astype(bool)
        total = p.sum() + t.sum()
        scores[name] = 1.0 if total == 0 else 2 * (p & t).sum() / total
    return pd.Series(scores)

### Train
For each fold, one patient group is held out for testing. From the rest, 10 patients are set aside for validation, which is used to pick the best epoch. Everyone else is used for training.

Each epoch prints how long it took, so after the first one you'll know roughly how long the whole run will take. Random left-right flips are added during training so the model learns that tumors can appear on either side of the brain.

In [ ]:
os.makedirs("models", exist_ok=True)
history = {}
all_test_scores = []

for fold in folds_to_run:
    test_patients = patient_groups[fold]
    other_patients = np.concatenate([g for k, g in enumerate(patient_groups) if k != fold])
    val_patients = other_patients[:10]
    train_patients = other_patients[10:]

    train_rows = slices["Patient"].isin(train_patients).values
    val_rows = slices["Patient"].isin(val_patients).values
    test_rows = slices["Patient"].isin(test_patients).values
    print(f"\nFold {fold}: {train_rows.sum()} train slices, {val_rows.sum()} val slices, {test_rows.sum()} test slices")

    train_images, train_masks = images[train_rows], masks[train_rows]
    val_images, val_masks = images[val_rows], masks[val_rows]
    val_names = slices["Patient"].values[val_rows]

    model = unet().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    best_dice = 0
    history[fold] = {"loss": [], "val_dice": []}

    for epoch in range(epochs):
        start = time.time()
        model.train()
        order = np.random.permutation(len(train_images))
        epoch_loss = 0

        for i in range(0, len(order), batch_size):
            batch = order[i:i + batch_size]
            x = torch.from_numpy(train_images[batch]).float().div(255)
            y = torch.from_numpy(train_masks[batch]).float()
            if np.random.rand() < 0.5:
                x = torch.flip(x, dims=[3])
                y = torch.flip(y, dims=[3])
            x, y = x.to(device), y.to(device)

            logits = model(x)
            loss = F.binary_cross_entropy_with_logits(logits, y) + dice_loss(logits, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(batch)

        epoch_loss /= len(order)
        val_dice = dice_per_patient(predict(model, val_images), val_masks, val_names).mean()
        history[fold]["loss"].append(epoch_loss)
        history[fold]["val_dice"].append(val_dice)

        if val_dice > best_dice:
            best_dice = val_dice
            torch.save(model.state_dict(), f"models/unet_fold{fold}.pt")
            note = "  <- Best so far, saved"
        else:
            note = ""
        print(f"Epoch {epoch + 1}/{epochs}  Loss {epoch_loss:.4f}  Val Dice {val_dice:.4f}  ({time.time() - start:.0f} s){note}")

    print(f"Fold {fold} best val Dice: {best_dice:.4f}")

### Save the training history
This saves the loss and validation Dice for every epoch to `training_history.csv`, so the training curves can be redrawn later without retraining.

In [ ]:
history_rows = []
for fold in history:
    for i in range(len(history[fold]["loss"])):
        history_rows.append({"Fold": fold, "Epoch": i + 1,
                             "Loss": history[fold]["loss"][i],
                             "Val Dice": history[fold]["val_dice"][i]})

pd.DataFrame(history_rows).to_csv("training_history.csv", index=False)
print("Saved training_history.csv")

### Training curves
Loss should go down and validation Dice should go up, then level off.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 7))
for fold in history:
    epochs_run = range(1, len(history[fold]["loss"]) + 1)
    axes[0].plot(epochs_run, history[fold]["loss"], linewidth=3, label=f"Fold {fold}")
    axes[1].plot(epochs_run, history[fold]["val_dice"], linewidth=3, label=f"Fold {fold}")

axes[0].set_title("Training loss")
axes[0].set_xlabel("Epoch")
axes[1].set_title("Validation Dice (per patient)")
axes[1].set_xlabel("Epoch")
axes[0].legend()
axes[1].legend()
plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()

### Test on held-out patients and save the predicted outlines
This loads the best model for each fold and runs it on that fold's test patients, who it has never seen. The predicted outlines are saved as image files in a `predictions` folder, which step 3 uses to measure tumor shape.

In [ ]:
rows_out = []

for fold in folds_to_run:
    model = unet().to(device)
    model.load_state_dict(torch.load(f"models/unet_fold{fold}.pt", map_location=device))

    test_rows = slices["Patient"].isin(patient_groups[fold]).values
    test_slices = slices[test_rows]
    test_predictions = predict(model, images[test_rows])

    scores = dice_per_patient(test_predictions, masks[test_rows], test_slices["Patient"].values)
    for name, score in scores.items():
        rows_out.append({"Patient": name, "Fold": fold, "Dice": score})

    for (_, row), prediction in zip(test_slices.iterrows(), test_predictions):
        folder = os.path.join("predictions", row["Folder"])
        os.makedirs(folder, exist_ok=True)
        name = os.path.basename(row["image_file"]).replace(".tif", "_pred.png")
        Image.fromarray(prediction[0] * 255).save(os.path.join(folder, name))

test_scores = pd.DataFrame(rows_out)
test_scores.to_csv("test_dice_scores.csv", index=False)

print("Test patients:", len(test_scores))
print("Mean Dice:", round(test_scores["Dice"].mean(), 4))
print("Median Dice:", round(test_scores["Dice"].median(), 4))

### Dice scores across test patients
Each bar counts patients. Most should sit toward the right. A few low scores are normal, usually for very small tumors where a small miss costs a lot.

In [ ]:
plt.figure(figsize=(12, 7))
plt.hist(test_scores["Dice"], bins=np.linspace(0, 1, 21), color="#5b2a86", edgecolor="white")
plt.axvline(test_scores["Dice"].mean(), color="black", linestyle="--", linewidth=3,
            label=f"Mean = {test_scores['Dice'].mean():.3f}")
plt.xlabel("Dice score per patient")
plt.ylabel("Patients")
plt.legend()
plt.tight_layout()
plt.savefig("test_dice_histogram.png", dpi=150)
plt.show()

### Example predictions
For the best, middle, and worst test patients, this shows the slice with the most tumor. Red is the expert's outline and yellow is the model's.

In [ ]:
ranked = test_scores.sort_values("Dice", ascending=False).reset_index(drop=True)
picks = [("Best", ranked.iloc[0]), ("Middle", ranked.iloc[len(ranked) // 2]), ("Worst", ranked.iloc[-1])]

fig, axes = plt.subplots(1, 3, figsize=(24, 9))
for ax, (label, pick) in zip(axes, picks):
    patient_rows = slices.index[slices["Patient"] == pick["Patient"]]
    biggest = patient_rows[masks[patient_rows].sum(axis=(1, 2, 3)).argmax()]
    row = slices.loc[biggest]

    pred_file = os.path.join("predictions", row["Folder"],
                             os.path.basename(row["image_file"]).replace(".tif", "_pred.png"))
    prediction = np.array(Image.open(pred_file)) > 0

    ax.imshow(images[biggest, 1], cmap="gray")
    ax.contour(masks[biggest, 0], colors="red", linewidths=2.5)
    if prediction.any():
        ax.contour(prediction, colors="yellow", linewidths=2.5)
    ax.set_title(f"{label}: {pick['Patient']}\nDice {pick['Dice']:.3f}", fontsize=20)
    ax.axis("off")

plt.tight_layout()
plt.savefig("example_predictions.png", dpi=150)
plt.show()

### Pack everything up for download
This zips the predicted outlines, trained models, Dice scores, and figures into one file. Download `project3_outputs.zip` from the Output tab and unzip it next to your other notebooks on your Mac.

In [ ]:
import shutil

os.makedirs("project3_outputs", exist_ok=True)
for name in ["predictions", "models"]:
    shutil.copytree(name, os.path.join("project3_outputs", name), dirs_exist_ok=True)
for name in ["test_dice_scores.csv", "training_history.csv", "slices_with_genetics2.csv", "training_curves.png",
             "test_dice_histogram.png", "example_predictions.png"]:
    shutil.copy(name, "project3_outputs")

shutil.make_archive("project3_outputs", "zip", "project3_outputs")
shutil.rmtree("project3_outputs")
print("Saved project3_outputs.zip")